# 第20章　眼科① ― 眼底写真と糖尿病網膜症**『本格実装 医療診断支援AI（実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-impl

## 20.3　マルチラベルという設計上の要点

```textRoot_Dataset/├── train/│   ├── images/                # 元画像（.jpg）│   └── masks/│       ├── MA/                # 微小血管瘤の二値マスク（.tif）│       ├── Hemo/              # 出血│       ├── HardExudate/       # 硬性白斑│       └── SoftExudate/       # 軟性白斑├── valid/                     # trainと同構成└── test/```

## 20.5　実装の骨格

In [ ]:
import torchimport torch.nn as nnimport segmentation_models_pytorch as smp# U-Netベースのマルチラベルセグメンテーションモデルmodel = smp.Unet(    encoder_name="efficientnet-b3",  # EfficientNetをエンコーダに    encoder_weights="imagenet",       # 事前学習の重みを利用    in_channels=3,                    # RGB入力    classes=4,                        # MA, Hemo, Hard, Soft の4病変    activation=None,                  # 出力は後でSigmoid（マルチラベル）)# Dice損失（マルチラベル）loss_fn = smp.losses.DiceLoss(mode="multilabel")# 病変ごとに独立したマスクを積み重ねて (4, H, W) の教師とするdef build_target(masks_dict):    channels = [masks_dict[k] for k in ["MA", "Hemo", "Hard", "Soft"]]    return torch.stack(channels, dim=0).float()

## スクリーニングの本丸 ― referable DR と国際重症度分類

In [ ]:
# 病変マップ→紹介判定の集約（説明可能なルート）と、直接グレーディング（高感度ルート）の併用def referable_decision(lesion_stats, grade_logits, refer_thr=0.35,                       heavy_hemo=20, moderate_or_worse=(2, 3, 4)):    # lesion_stats: {"MA":count, "Hemo":count, "Hard":.., "Soft":..} 病変ごとの検出数/面積    # moderate_or_worse: ICDR 0〜4 のうち中等症NPDR以上に当たるクラスindex    # 注意：これは4-2-1ルールそのものではない。4-2-1は象限別の条件なので、    # 実装するには象限ごとの病変数が要る。ここは「多発出血による簡易な紹介トリガー」。    hemorrhage_heavy = lesion_stats["Hemo"] >= heavy_hemo    # 眼底全体での出血数    macular_exudate  = lesion_stats["Hard_at_macula"] > 0    # 黄斑近傍の硬性白斑＝DME疑い    # softmax はクラス軸だけで取り、sum もクラス軸で閉じる。全体を sum すると    # バッチの複数患者を足し込んでしまう（この関数は1症例ぶんの入力を前提とする）。    grade_prob = grade_logits.softmax(-1)[..., moderate_or_worse].sum(-1)    refer = (grade_prob >= refer_thr) or hemorrhage_heavy or macular_exudate    return {"refer": bool(refer), "grade_prob": float(grade_prob),            "reason": {"heavy_hemorrhage": hemorrhage_heavy, "dme_suspect": macular_exudate}}

## 追加ケース ― 緑内障スクリーニングと視神経乳頭の定量

In [ ]:
def vertical_cdr(disc_mask, cup_mask):    def v_extent(m):        rows = m.any(axis=1)                    # 垂直方向に存在する行        return rows.sum()                       # 縦径（ピクセル）    return v_extent(cup_mask) / max(v_extent(disc_mask), 1)   # 垂直CDR# CDR に加え、リム(disc-cup)の厚みが ISNT則（下≧上≧鼻≧耳）を満たすかも見る